# Ch 10 · Domain Randomization with Batched Model Fields

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/master/dpdk-research/mujoco-warp-course/notebooks/10-domain-randomization.ipynb)

Companion notebook for the chapter page `chapters/10-domain-randomization.html` ([page](../chapters/10-domain-randomization.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

**How this notebook works.** Every example cell below is the same file as on the chapter page (`examples/10-domain-randomization/`). Budgets follow `ON_GPU`: on a Colab GPU the examples use thousands of worlds, on a CPU runtime a handful. The next cell writes the running helper module `my_env.py` (the course project file, snapshot after Chapter 10: `make_worlds`, `reset_done`, `nonfinite_reset`, `snapshot`/`restore`, `describe_overflow`, and the new `randomize`) so the notebook runs in an empty `/content`.

In [ ]:
%%writefile my_env.py
"""my_env.py - the course's running helper module, snapshot after Chapter 10.

Ch 6 added make_worlds. Ch 8 added reset_done, nonfinite_reset, snapshot and restore.
Ch 9 added describe_overflow (Example 9.7).
Ch 10 adds randomize (Example 10.9): in-place, graph-friendly friction re-randomization.
Later chapters extend this file (Ch 14 sensor_slices, Ch 24 capture_step,
Ch 26 obs/reward kernels, Ch 30 the environments).
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

__all__ = ["make_worlds", "reset_done", "nonfinite_reset", "snapshot", "restore",
           "describe_overflow", "randomize"]


# --- Ch 6 ----------------------------------------------------------------------------------------
def make_worlds(xml, nworld, nconmax=None, njmax=None, batch_sizes=None, **sizes):
    """Compile `xml` and return (mjm, m, d) with `nworld` worlds (Ch 6 signature).

    nconmax / njmax are per world; batch_sizes (e.g. {"body_mass": nworld}) gives chosen Model
    fields their own rows (Ch 10); any other keyword (naconmax=, ...) goes to make_data.
    """
    mjm = mujoco.MjModel.from_xml_string(xml)
    m = mjw.put_model(mjm, batch_sizes=batch_sizes)
    d = mjw.make_data(mjm, nworld=nworld, nconmax=nconmax, njmax=njmax, **sizes)
    assert d.qpos.shape == (nworld, mjm.nq), d.qpos.shape
    return mjm, m, d


# --- Ch 8: resets --------------------------------------------------------------------------------
@wp.kernel
def _nonfinite_kernel(qpos_in: wp.array2d[float], qvel_in: wp.array2d[float],
                      done_out: wp.array[bool]):
    worldid = wp.tid()
    bad = bool(False)
    for i in range(qpos_in.shape[1]):
        if not wp.isfinite(qpos_in[worldid, i]):
            bad = True
    for i in range(qvel_in.shape[1]):
        if not wp.isfinite(qvel_in[worldid, i]):
            bad = True
    if bad:
        done_out[worldid] = True          # only ever sets flags; never clears another reason


def reset_done(m, d, done):
    """Reset the worlds whose done flag is True, then recompute derived quantities.

    `done` is a wp.array of bool with shape (nworld,). Everything stays on the device, so this
    can be captured in a CUDA graph together with mjw.step (Ch 24, Ch 26).
    """
    mjw.reset_data(m, d, reset=done)
    mjw.forward(m, d)                     # reset_data does not recompute xpos, sensors, ...


def nonfinite_reset(m, d, done):
    """OR `done` with "qpos or qvel has a NaN/Inf" (MJWarp has no mj_checkPos/mj_checkVel)."""
    wp.launch(_nonfinite_kernel, dim=d.nworld, inputs=[d.qpos, d.qvel], outputs=[done])


# --- Ch 8: state snapshots -----------------------------------------------------------------------
def snapshot(m, d, mjm, sig, buf=None):
    """Copy the state components in `sig` of every world into a (nworld, size) float array.

    Use State.PHYSICS for MPC (it includes HISTORY and PLASTIC; note it has no TIME bit),
    add State.WARMSTART for bitwise replays with contacts, State.INTEGRATION for everything.
    """
    size = mujoco.mj_stateSize(mjm, int(sig))
    if buf is None:
        buf = wp.zeros((d.nworld, size), dtype=float)
    # get_state does NOT check the size: an undersized buffer is written out of bounds.
    assert buf.shape == (d.nworld, size), f"state buffer {buf.shape} != {(d.nworld, size)}"
    mjw.get_state(m, d, buf, int(sig))
    return buf


def restore(m, d, mjm, buf, sig, active=None):
    """Write a snapshot back (only where the per-world bool mask `active` is True, if given)."""
    assert buf.shape == (d.nworld, mujoco.mj_stateSize(mjm, int(sig)))
    mjw.set_state(m, d, buf, int(sig), active)


# --- Ch 9: overflow ------------------------------------------------------------------------------
def describe_overflow(d, max_worlds=8):
    """Decode d.overflow: "none", or one line per flagged world plus a world count per bit.

    Loops over mjw.OverflowType (Python >= 3.11 iterates the single-bit members only), so bits
    added by a later MJWarp release are decoded without editing this function.
    """
    ov = d.overflow.numpy()
    if not ov.any():
        return "none"
    lines = []
    flagged = np.nonzero(ov)[0]
    for w in flagged[:max_worlds]:
        names = [bit.name for bit in mjw.OverflowType if ov[w] & bit]
        lines.append(f"world {w}: " + " | ".join(names))
    if len(flagged) > max_worlds:
        lines.append(f"... and {len(flagged) - max_worlds} more flagged worlds")
    counts = [f"{bit.name} x{int(np.count_nonzero(ov & bit))}" for bit in mjw.OverflowType
              if (ov & bit).any()]
    lines.append("worlds per bit: " + ", ".join(counts))
    return "\n".join(lines)


# --- Ch 10: domain randomization ----------------------------------------------------------------
@wp.kernel
def _randomize_friction_kernel(
    seed: wp.array(dtype=int),          # device array: a graph replays with the CURRENT value
    episode: wp.array(dtype=int),       # per-world episode counter, bumped on every re-draw
    reset: wp.array(dtype=bool),        # which worlds to re-draw (all if every entry is True)
    lo: float,
    hi: float,
    geom_friction: wp.array2d(dtype=wp.vec3),   # m.geom_friction, shape (nworld, ngeom)
):
    worldid, geomid = wp.tid()
    if not reset[worldid]:
        return
    # one random stream per (seed, world, episode): reproducible and different every episode
    rng = wp.rand_init(seed[0], worldid * 7919 + episode[worldid] * 104729)
    mu = wp.randf(rng, lo, hi)
    f = geom_friction[worldid, geomid]
    geom_friction[worldid, geomid] = wp.vec3(mu, f[1], f[2])


@wp.kernel
def _bump_episode_kernel(reset: wp.array(dtype=bool), episode: wp.array(dtype=int)):
    worldid = wp.tid()
    if reset[worldid]:
        episode[worldid] = episode[worldid] + 1


def randomize(m, seed, episode, reset, lo=0.1, hi=1.0):
    """Draw a new sliding friction in [lo, hi) for every geom of the worlds marked in reset.

    Writes m.geom_friction in place (it must already have one row per world, e.g. from
    put_model(batch_sizes={"geom_friction": nworld})), so a captured CUDA graph that contains
    this call keeps working: only array CONTENTS change, never array objects.
    seed: wp.array(dtype=int, shape=(1,)); episode: wp.array(dtype=int, shape=(nworld,));
    reset: wp.array(dtype=bool, shape=(nworld,)).
    """
    nworld = reset.shape[0]
    assert m.geom_friction.shape[0] == nworld, "batch geom_friction to nworld first"
    wp.launch(_randomize_friction_kernel, dim=(nworld, m.ngeom),
              inputs=[seed, episode, reset, lo, hi], outputs=[m.geom_friction])
    wp.launch(_bump_episode_kernel, dim=nworld, inputs=[reset], outputs=[episode])

Cells follow the order of the chapter page. Example numbers follow the example files (other chapters cite them), so they are not always in sequence.

## 1. Simple DR: opt arrays (replace them)
Per-world fluid density. Note the `m.has_fluid` host flag: it is decided from the XML, so the XML needs a non-zero density for the per-world values to matter. The inertia-box model adds drag only (no buoyancy).

In [ ]:
# Example 10.2 - ex2_density_worlds.py
"""Example 10.2 - per-world fluid density: vacuum, air and water in one batch.

Ch 6 ex7 gave every world its own gravity by REPLACING m.opt.gravity. Here we do the same with a
field Ch 6 did not touch: m.opt.density. MuJoCo's inertia-box fluid model then adds a quadratic
drag F_i = -0.5 * density * A_i * |v_i| * v_i on each axis of the body's inertia box, with A_x = b_y * b_z
etc. (passive.py::_fluid_force; this model has no buoyancy term), so the "water" world falls much more slowly. No floor: we only watch the ball.
Run: python ex2_density_worlds.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

# density="1.2" in the XML matters: put_model sets the host flag m.has_fluid from the XML values,
# and the fluid kernels only run when that flag is True (see the second half of this script).
LIGHT_BALL = """
<mujoco model="light_ball">
  <option timestep="0.01" density="{density}"/>
  <worldbody>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom type="sphere" size=".1" mass="4"/>
    </body>
  </worldbody>
</mujoco>
"""
DENSITY = [0.0, 1.2, 1000.0]          # vacuum, air, water [kg/m^3]
NSTEP = 100                            # 1 s of simulated time


def run(xml_density):
    mjm = mujoco.MjModel.from_xml_string(LIGHT_BALL.format(density=xml_density))
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=len(DENSITY))
    m.opt.density = wp.array(DENSITY, dtype=float)        # replace: batch of 3, one per world
    for _ in range(NSTEP):
        mjw.step(m, d)
    return m, d.qpos.numpy()[:, 2], d.qvel.numpy()[:, 2]


m, z, vz = run(1.2)
print("m.opt.density:", m.opt.density.numpy(), "| m.has_fluid:", m.has_fluid)
for rho, zi, vi in zip(DENSITY, z, vz):
    print(f"  density {rho:7.1f}  ->  z after 1 s = {zi:+.4f} m, vz = {vi:+.3f} m/s")
print("free fall from 1 m for 1 s would give z =", round(1 - 0.5 * 9.81, 4))
# inertia box of a solid sphere: side = sqrt(6 * 0.4 r^2) = 0.155 m, frontal area A = side^2
side = np.sqrt(6 * 0.4 * 0.1**2)
print(f"water terminal speed sqrt(2 m g / (rho A)) = {np.sqrt(2 * 4 * 9.81 / (1000 * side**2)):.3f} m/s")

m0, z0, _ = run(0.0)
print("same batch, but XML density=0 -> m.has_fluid:", m0.has_fluid, "| z:", z0)

Per-world timestep: after 10 steps every world is at a different `d.time`.

In [ ]:
# Example 10.3 - ex3_timestep_worlds.py
"""Example 10.3 - a different timestep in every world.

m.opt.timestep is a "*" array, so we replace it with one value per world. Every kernel that
needs dt reads opt_timestep[worldid % opt_timestep.shape[0]] (forward.py::_next_time and the
integrators), so after the same number of steps each world is at a different time.
Run: python ex3_timestep_worlds.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=3, nconmax=4, njmax=16)

print("before: m.opt.timestep =", m.opt.timestep.numpy(), "shape", m.opt.timestep.shape)
m.opt.timestep = wp.array([0.001, 0.01, 0.02], dtype=float)   # one per world
print("after:  m.opt.timestep =", m.opt.timestep.numpy(), "shape", m.opt.timestep.shape)

for _ in range(10):
    mjw.step(m, d)
print("d.time after 10 steps:", d.time.numpy())
print("ball z after 10 steps:", d.qpos.numpy()[:, 2])
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## 2. Friction per world
The same box on a 30 degree slope with friction 0.1 ... 1.0. Contact friction is the element-wise MAX of the two geoms, so the example sets box and slope to the same value. Compare with Coulomb: `s = 0.5 g (sin - mu cos) t^2`.

In [ ]:
# Example 10.4 - ex4_friction_slide.py
"""Example 10.4 - per-world friction: the same box slides down the same slope NWORLD ways.

geom_friction is a "*" field. We allocate one row per world with put_model(batch_sizes=...) and
write a friction in [0.1, 1.0] into every row, dealt out to the worlds in shuffled order
(as a random DR draw would be). The contact kernels read
geom_friction[worldid % geom_friction.shape[0], geom] (collision_core.py::contact_material_params)
and mix the two geoms with an element-wise MAX, so we set the box AND the plane to the same value.
Run: python ex4_friction_slide.py [--dump-json PATH]
  --dump-json PATH writes decimated per-world slide distances for the page's DR mosaic (Rule D).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=3, suppress=True)

SLOPE_DEG = 30.0
n = np.array([np.sin(np.radians(SLOPE_DEG)), 0.0, np.cos(np.radians(SLOPE_DEG))])  # plane normal
p0 = 0.101 * n                                    # box centre: half-size + 1 mm above the plane
SLOPE = f"""
<mujoco model="slope">
  <option timestep="0.005"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="slope" type="plane" size="5 5 .1" euler="0 {SLOPE_DEG} 0"/>
    <body name="box" pos="{p0[0]:.5f} 0 {p0[2]:.5f}" euler="0 {SLOPE_DEG} 0">
      <freejoint/>
      <geom name="box" type="box" size=".1 .1 .1" mass="1" rgba=".3 .3 1 1"/>
    </body>
  </worldbody>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 16
NSTEP = 200                                       # 1 s

mjm = mujoco.MjModel.from_xml_string(SLOPE)
m = mjw.put_model(mjm, batch_sizes={"geom_friction": NWORLD})
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=8, njmax=32)

mu = np.random.default_rng(0).permutation(np.linspace(0.1, 1.0, NWORLD))   # shuffled over worlds
fr = m.geom_friction.numpy()                      # (NWORLD, ngeom, 3): slide, spin, roll
fr[:, :, 0] = mu[:, None]                         # same sliding friction for slope and box
m.geom_friction.assign(fr)
print("geom_friction shape:", m.geom_friction.shape)

x0 = d.qpos.numpy()[:, :3].copy()
frames = []
for i in range(NSTEP):
    mjw.step(m, d)
    if args.dump_json and i % 5 == 4:             # decimate to 40 frames (reads to host: demo only)
        frames.append([round(float(v), 3) for v in
                       np.linalg.norm(d.qpos.numpy()[:, :3] - x0, axis=1)])
wp.synchronize()

dist = np.linalg.norm(d.qpos.numpy()[:, :3] - x0, axis=1)
tan = np.tan(np.radians(SLOPE_DEG))
print(f"slope {SLOPE_DEG:.0f} deg: a box slides when mu < tan(slope) = {tan:.3f}")
for w in range(0, NWORLD, max(1, NWORLD // 16)):
    pred = max(0.0, 0.5 * 9.81 * (np.sin(np.radians(SLOPE_DEG)) - mu[w] * np.cos(np.radians(SLOPE_DEG))))
    print(f"  world {w:4d}  mu = {mu[w]:.2f}  slide = {dist[w]:.3f} m   (Coulomb: {pred:.3f} m)")
order = np.argsort(mu)                            # worlds sorted by friction
print("monotone (non-increasing) in mu:", bool(np.all(np.diff(dist[order]) <= 1e-3)))
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": NWORLD, "dt": 5 * float(mjm.opt.timestep), "kind": "slide",
                   "slope_deg": SLOPE_DEG, "labels": [f"mu={v:.2f}" for v in mu],
                   "mu": [round(float(v), 3) for v in mu], "frames": frames},
                  f, separators=(",", ":"))
    print("wrote", args.dump_json, len(frames), "frames")

## 3. Actuator strength: gear and gain
No derived constants are needed for motors.

In [ ]:
# Example 10.8 - ex8_actuator_gain.py
"""Example 10.8 - randomize actuator strength: actuator_gear or actuator_gainprm per world.

A motor's force on its joint is gear * gain * ctrl (gain = gainprm[0] for a FIXED gain). Both
fields are "*" arrays, so the same ctrl pushes the cart harder in some worlds than in others,
like motors from a production line that are all a little different. No set_const is needed for
the dynamics (transmission reads actuator_gear[worldid % B] every step); actuator_acc0 is only
used by muscles (Example 10.12).
Run: python ex8_actuator_gain.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

CARTPOLE = """
<mujoco model="cartpole">
  <option timestep="0.01" integrator="implicitfast"/>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="front" pos="0 -4 1" xyaxes="1 0 0 0 0 1"/>
    <geom name="rail" type="capsule" fromto="-2 0 1 2 0 1" size=".02" contype="0" conaffinity="0"/>
    <body name="cart" pos="0 0 1">
      <joint name="slider" type="slide" axis="1 0 0" range="-1.8 1.8" limited="true" damping="0.1"/>
      <geom name="cart" type="box" size=".2 .1 .05" mass="1" contype="0" conaffinity="0"/>
      <body name="pole">
        <joint name="hinge" type="hinge" axis="0 1 0" damping="0.01"/>
        <geom name="pole" type="capsule" fromto="0 0 0 0 0 .6" size=".03" mass="0.1"
              contype="0" conaffinity="0"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="push" joint="slider" gear="10" ctrlrange="-1 1" ctrllimited="true"/></actuator>
  <sensor>
    <jointpos joint="slider"/><jointpos joint="hinge"/>
    <jointvel joint="slider"/><jointvel joint="hinge"/>
  </sensor>
</mujoco>
"""
NWORLD = 4
mjm = mujoco.MjModel.from_xml_string(CARTPOLE)

# A. gear per world: 5, 10, 15, 20
m = mjw.put_model(mjm, batch_sizes={"actuator_gear": NWORLD})
d = mjw.make_data(mjm, nworld=NWORLD)
gear = m.actuator_gear.numpy()                    # (NWORLD, nu, 6): spatial gear, [0] for joints
gear[:, 0, 0] = [5.0, 10.0, 15.0, 20.0]
m.actuator_gear.assign(gear)
d.ctrl.fill_(0.5)
mjw.forward(m, d)
print("A. gear       =", m.actuator_gear.numpy()[:, 0, 0])
print("   force      =", d.actuator_force.numpy()[:, 0], "(gain * ctrl, before gear)")
print("   qfrc_actuator[slider] =", d.qfrc_actuator.numpy()[:, 0])
print("   cart qacc  =", d.qacc.numpy()[:, 0])

# B. gain per world, gear fixed at 10: gainprm[0] = 0.5, 1, 1.5, 2
m = mjw.put_model(mjm, batch_sizes={"actuator_gainprm": NWORLD})
d = mjw.make_data(mjm, nworld=NWORLD)
gain = m.actuator_gainprm.numpy()                 # (NWORLD, nu, 10)
gain[:, 0, 0] = [0.5, 1.0, 1.5, 2.0]
m.actuator_gainprm.assign(gain)
d.ctrl.fill_(0.5)
mjw.forward(m, d)
print("B. gainprm[0] =", m.actuator_gainprm.numpy()[:, 0, 0])
print("   force      =", d.actuator_force.numpy()[:, 0])
print("   cart qacc  =", d.qacc.numpy()[:, 0], "(same as A: gear * gain is what counts)")

## 4. What can differ per world?
`put_model` tags every array whose type spec starts with `"*"` with `_is_batched = True`. Example 10.1 lists them, grouped by prefix. Numeric parameters can vary; sizes, topology, types and discrete options cannot.

In [ ]:
# Example 10.1 - ex1_list_batchable.py
"""Example 10.1 - which Model fields can differ per world? List every "*"-batched field.

put_model marks every array whose type spec starts with "*" (Model) or "nworld" (Data) with
arr._is_batched = True (warp_util.mark_batched). Those are the fields you may give one row per
world. Everything else (topology, ids, sizes, discrete options) is shared by all worlds.
Run: python ex1_list_batchable.py [--dump-json PATH]
"""
import argparse
import collections
import dataclasses
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

# any model works; the humanoid has bodies, joints, actuators, cameras and lights
path = epath.resource_path("mujoco_warp") / "test_data/humanoid/humanoid.xml"
mjm = mujoco.MjModel.from_xml_path(str(path))
m = mjw.put_model(mjm)

batched, shared = [], []
for f in dataclasses.fields(m):
    arr = getattr(m, f.name)
    if isinstance(arr, wp.array):
        (batched if getattr(arr, "_is_batched", False) else shared).append(f.name)
opt_batched = [f.name for f in dataclasses.fields(m.opt)
               if getattr(getattr(m.opt, f.name), "_is_batched", False)]

groups = collections.defaultdict(list)
for name in batched:
    groups[name.split("_")[0]].append(name)

print(f"Model arrays: {len(batched)} batched ('*'), {len(shared)} shared")
for prefix, names in sorted(groups.items(), key=lambda kv: -len(kv[1])):
    print(f"  {prefix:9s} {len(names):3d}  e.g. {', '.join(names[:3])}")
print(f"m.opt batched arrays ({len(opt_batched)}):", ", ".join(opt_batched))
print("every batched array starts with batch size 1:",
      all(getattr(m, n).shape[0] == 1 for n in batched))
print("some shared arrays:", ", ".join(n for n in ["geom_type", "body_parentid", "jnt_type",
                                                    "dof_jntid", "geom_bodyid"] if n in shared))
print("plain ints in m.opt (one value for all worlds):",
      {k: int(getattr(m.opt, k)) for k in ["integrator", "cone", "solver", "iterations",
                                            "ls_iterations"]})

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"groups": {k: v for k, v in sorted(groups.items())}, "opt": opt_batched,
                   "nbatched": len(batched), "nshared": len(shared)}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 5. Mass and set_const
Batch the inputs (`body_mass`, `body_inertia`) **and** the derived outputs (`body_subtreemass`, `body_invweight0`, `dof_invweight0`, `actuator_acc0`), then call `mjw.set_const(m, d)` (groups=ALL).

In [ ]:
# Example 10.5 - ex5_mass_set_const.py
"""Example 10.5 - per-world mass, done right: batch the inputs AND the derived outputs.

body_mass feeds derived constants that put_model computed once on the host (body_subtreemass,
dof_invweight0, body_invweight0, actuator_acc0). mjw.set_const recomputes them on the device,
one row per world, but only if those output fields also have one row per world.
Run: python ex5_mass_set_const.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

# a box on a rail (slide joint) carrying a 0.5 kg payload, pushed by a motor
SLIDER_BOX = """
<mujoco model="slider_box">
  <option timestep="0.01"/>
  <worldbody>
    <body name="box" pos="0.5 0 0.5">
      <joint name="rail" type="slide" axis="1 0 0"/>
      <geom type="box" size=".1 .1 .1" mass="1" contype="0" conaffinity="0"/>
      <body name="payload" pos="0 0 .15">
        <geom type="sphere" size=".05" mass="0.5" contype="0" conaffinity="0"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="push" joint="rail" gear="1"/></actuator>
</mujoco>
"""
NWORLD = 4
INPUTS = ["body_mass", "body_inertia"]                    # what we randomize
DERIVED = ["body_subtreemass", "body_invweight0", "dof_invweight0", "actuator_acc0"]

mjm = mujoco.MjModel.from_xml_string(SLIDER_BOX)
box = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, "box")
m = mjw.put_model(mjm, batch_sizes={k: NWORLD for k in INPUTS + DERIVED})
d = mjw.make_data(mjm, nworld=NWORLD)

scale = np.array([1.0, 2.0, 4.0, 8.0])                    # box mass 1, 2, 4, 8 kg
bm = m.body_mass.numpy()
bm[:, box] *= scale
m.body_mass.assign(bm)
bi = m.body_inertia.numpy()
bi[:, box] *= scale[:, None]                               # inertia = sum(m r^2): scale together
m.body_inertia.assign(bi)

print("before set_const: body_subtreemass[:, box] =", m.body_subtreemass.numpy()[:, box])
mjw.set_const(m, d)                                        # groups=SetConstGroup.ALL
print("after  set_const: body_subtreemass[:, box] =", m.body_subtreemass.numpy()[:, box])
print("actuator_acc0 (unit motor force -> accel)  =", m.actuator_acc0.numpy()[:, 0])
print("1 / (box + payload mass)                    =", 1.0 / (scale + 0.5))
print("dof_invweight0 [1/kg]                       =", m.dof_invweight0.numpy()[:, 0])

d.ctrl.fill_(1.0)                                          # 1 N in every world
mjw.forward(m, d)
print("qacc for ctrl = 1 N                         =", d.qacc.numpy()[:, 0])

The pitfall: batch only the inputs and `set_const` writes one shared row, computed from world 0. `qacc` is still right, but `subtree_com` divides by the wrong mass.

In [ ]:
# Example 10.6 - ex6_derived_pitfall.py
"""Example 10.6 - the derived-field pitfall: batch body_mass only, and set_const writes ONE row.

Same model and masses as Example 10.5, but the derived outputs keep batch size 1. set_const
launches its kernels over the OUTPUT's leading size (e.g. dim=(body_subtreemass.shape[0], nbody)),
so it computes world 0 only and every world then reads that single row.
Run: python ex6_derived_pitfall.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

SLIDER_BOX = """
<mujoco model="slider_box">
  <option timestep="0.01"/>
  <worldbody>
    <body name="box" pos="0.5 0 0.5">
      <joint name="rail" type="slide" axis="1 0 0"/>
      <geom type="box" size=".1 .1 .1" mass="1" contype="0" conaffinity="0"/>
      <body name="payload" pos="0 0 .15">
        <geom type="sphere" size=".05" mass="0.5" contype="0" conaffinity="0"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="push" joint="rail" gear="1"/></actuator>
</mujoco>
"""
NWORLD = 4

mjm = mujoco.MjModel.from_xml_string(SLIDER_BOX)
box = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, "box")
m = mjw.put_model(mjm, batch_sizes={"body_mass": NWORLD, "body_inertia": NWORLD})  # inputs only!
d = mjw.make_data(mjm, nworld=NWORLD)

scale = np.array([1.0, 2.0, 4.0, 8.0])
bm = m.body_mass.numpy()
bm[:, box] *= scale
m.body_mass.assign(bm)
bi = m.body_inertia.numpy()
bi[:, box] *= scale[:, None]
m.body_inertia.assign(bi)
mjw.set_const(m, d)

print("body_mass[:, box]        shape", m.body_mass.shape, "->", m.body_mass.numpy()[:, box])
print("body_subtreemass[:, box] shape", m.body_subtreemass.shape, "->",
      m.body_subtreemass.numpy()[:, box], "(world 0's value, shared)")
print("actuator_acc0            shape", m.actuator_acc0.shape, "->", m.actuator_acc0.numpy()[:, 0])

d.ctrl.fill_(1.0)
mjw.forward(m, d)
print("qacc per world (uses body_mass directly, still right):", d.qacc.numpy()[:, 0])
# subtree_com = sum(xipos * body_mass) / body_subtreemass: the stale denominator shows up here
print("subtree_com x of 'box' (true value 0.5 in every world):", d.subtree_com.numpy()[:, box, 0])

## 6. Sizes: geom_size + bounds, set_const(FIXED)
Forget `geom_rbound`/`geom_aabb` and the broadphase prunes contacts of the bigger ball. Calling `set_const(FIXED)` without batching the bounds does not help: `_set_bound` launches over the largest batch size, so every world writes the one shared row (run 2b). The last lines show the DeprecationWarning of the old alias `set_const_fixed`, once.

In [ ]:
# Example 10.7 - ex7_size_fixed.py
"""Example 10.7 - per-world ball radius: geom_size plus its bounds, then set_const(FIXED).

geom_size changes the collision shape, but the broadphase prunes pairs with geom_rbound and
geom_aabb, which put_model computed from the XML size. Batch all three and recompute the bounds
with mjw.set_const(m, d, groups=mjw.SetConstGroup.FIXED). The second half calls the deprecated
alias set_const_fixed once, only to show its DeprecationWarning. Run 2b shows that forgetting the
bounds is not fixed by set_const: all worlds then race to write the one shared bounds row.
Run: python ex7_size_fixed.py
"""
import warnings

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""
RADII = np.array([0.05, 0.1, 0.2])
NW = len(RADII)
NSTEP = 1000                                      # 2 s: the balls drop and come to rest


def make(fields):
    mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
    m = mjw.put_model(mjm, batch_sizes={k: NW for k in fields})
    d = mjw.make_data(mjm, nworld=NW, nconmax=4, njmax=16)
    ball = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_GEOM, "ball")
    gs = m.geom_size.numpy()
    gs[:, ball, 0] = RADII                        # sphere: size[0] is the radius
    m.geom_size.assign(gs)
    return mjm, m, d, ball


def rest_height(m, d):
    for _ in range(NSTEP):
        mjw.step(m, d)
    return d.qpos.numpy()[:, 2]


# 1. the right way: size + bounds batched, bounds recomputed
mjm, m, d, ball = make(["geom_size", "geom_rbound", "geom_aabb"])
print("geom_rbound before:", m.geom_rbound.numpy()[:, ball])
mjw.set_const(m, d, groups=mjw.SetConstGroup.FIXED)
print("geom_rbound after: ", m.geom_rbound.numpy()[:, ball])
print("geom_aabb half-size after:", m.geom_aabb.numpy()[:, ball, 1, 0])
print("rest z (expect r - ~0.0004 of soft-contact penetration):", rest_height(m, d))
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

# 2. what happens if you forget the bounds: rbound stays 0.1 for every world
_, m2, d2, _ = make(["geom_size"])
print("forgot the bounds, rbound =", m2.geom_rbound.numpy()[:, ball], "-> rest z:",
      rest_height(m2, d2))

# 2b. forgot the bounds AND called set_const(FIXED): _set_bound launches over the LARGEST batch
# size of its inputs and outputs (here 3), so all three worlds write the one shared row (a race;
# on CPU the last world wins, on a GPU the winner is unspecified)
_, m3, d3, _ = make(["geom_size"])
mjw.set_const(m3, d3, groups=mjw.SetConstGroup.FIXED)
print("forgot the bounds + FIXED, shared rbound =", m3.geom_rbound.numpy()[:, ball],
      "aabb half-size =", m3.geom_aabb.numpy()[:, ball, 1, 0])

# 3. the deprecated alias, once (never use it in new code)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    mjw.set_const_fixed(m, d)
for w in caught:
    print(f"{w.category.__name__}: {w.message}")

## 7. Actuator length ranges after gear DR
`set_length_range` copies jnt/tendon range x gear for limited transmissions and writes (0, 0) for everything else (its `index` argument is ignored in 4.0.1). Batch `actuator_lengthrange` to nworld: the kernel writes `[worldid, actuator]` without the `% batch` wrap.

In [ ]:
# Example 10.12 - ex12_length_range.py
"""Example 10.12 - actuator_lengthrange after gear randomization: mjw.set_length_range.

A muscle's force-length curve is defined over actuator_lengthrange (Ch 12). If you randomize the
gear (or the joint/tendon range), the length range must follow. In mujoco-warp 4.0.1
set_length_range copies jnt_range or tendon_range * gear for LIMITED joint and tendon
transmissions; every other actuator gets (0, 0), even if the XML gave it a range.
We load test_data/actuation/muscle.xml (one muscle on an unlimited tendon) and add, with MjSpec,
a range to its slide joint and a second muscle that drives that joint.
Run: python ex12_length_range.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

np.set_printoptions(precision=4, suppress=True)

path = epath.resource_path("mujoco_warp") / "test_data/actuation/muscle.xml"
spec = mujoco.MjSpec.from_file(str(path))
joint = spec.joints[0]
joint.name = "slide"
joint.limited = mujoco.mjtLimited.mjLIMITED_TRUE
joint.range = [-0.25, 0.25]
act = spec.add_actuator()
act.name = "joint_muscle"
act.trntype = mujoco.mjtTrn.mjTRN_JOINT
act.target = "slide"
act.set_to_muscle(tausmooth=0.0)
mjm = spec.compile()

NWORLD = 3
# set_length_range launches dim=(nworld, nu) and writes actuator_lengthrange[worldid, actuator]
# WITHOUT the "% batch" wrap, so give the output one row per world.
m = mjw.put_model(mjm, batch_sizes={"actuator_gear": NWORLD, "actuator_lengthrange": NWORLD})
d = mjw.make_data(mjm, nworld=NWORLD)

print("actuators: 0 = tendon muscle (tendon unlimited), 1 = joint muscle (range +-0.25)")
print("lengthrange from the compiler (world 0):\n", m.actuator_lengthrange.numpy()[0])
gear = m.actuator_gear.numpy()
gear[:, :, 0] *= np.array([1.0, 2.0, 3.0])[:, None]        # gear x1, x2, x3 per world
m.actuator_gear.assign(gear)
mjw.set_length_range(m, d)
lr = m.actuator_lengthrange.numpy()
for w in range(NWORLD):
    print(f"world {w}: gear {gear[w, 1, 0]:.0f}  tendon muscle {lr[w, 0]}  joint muscle {lr[w, 1]}")

## 8. What batch_sizes refuses
`m.opt.*` fields can only be replaced; structural ints cannot be batched.

In [ ]:
# Example 10.10 - ex10_batch_size_errors.py
"""Example 10.10 - what put_model(batch_sizes=...) refuses, and the fix for m.opt fields.

batch_sizes keys must be top-level Model fields whose type spec starts with "*" (io.py::put_model).
m.opt.* arrays are "*" too, but they live in the nested Option dataclass, so they can only be
replaced. Integer topology fields such as geom_type are shared and cannot be batched at all.
Run: python ex10_batch_size_errors.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""
mjm = mujoco.MjModel.from_xml_string(BALL_DROP)

for request in ({"opt.timestep": 4}, {"geom_type": 4}, {"nbody": 4}, {"body_mass": 0}):
    try:
        mjw.put_model(mjm, batch_sizes=request)
        print(f"{request}: accepted")
    except ValueError as e:
        print(f"{request}: ValueError: {e}")

# the fix for opt fields: replace the array after put_model
m = mjw.put_model(mjm)
m.opt.timestep = wp.array(np.full(4, 0.002), dtype=float)
print("replaced m.opt.timestep:", m.opt.timestep.shape)
print("geom_type stays shared:", m.geom_type.shape, m.geom_type.numpy())

## 9. Re-randomize on device at episode resets
An in-place kernel with a device seed, a per-world episode counter and a reset mask. The second half captures the reset path in a graph and replays it with two seeds. This is `my_env.randomize`.

In [ ]:
# Example 10.9 - ex9_rerandomize_kernel.py
"""Example 10.9 - re-randomize friction on the device at episode resets (graph friendly).

A Warp kernel writes new sliding frictions into m.geom_friction IN PLACE, one random stream per
(seed, world, episode) from wp.rand_init / wp.randf. Nothing is replaced and nothing is copied
from the host, so the same code can live inside a captured CUDA graph. The seed is a device
array: a graph stores array pointers, so a replay reads the seed's CURRENT value. (A plain
Python int would be frozen into the graph at capture time.) This is my_env.randomize.
Run: python ex9_rerandomize_kernel.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=3, suppress=True)

SLOPE = """
<mujoco model="slope">
  <option timestep="0.005"/>
  <worldbody>
    <geom name="slope" type="plane" size="5 5 .1" euler="0 30 0"/>
    <body name="box" pos="0.0505 0 0.08747" euler="0 30 0">
      <freejoint/>
      <geom name="box" type="box" size=".1 .1 .1" mass="1"/>
    </body>
  </worldbody>
</mujoco>
"""


@wp.kernel
def _randomize_friction_kernel(
    seed: wp.array(dtype=int),          # device array: a graph replays with the CURRENT value
    episode: wp.array(dtype=int),       # per-world episode counter, bumped on every re-draw
    reset: wp.array(dtype=bool),        # which worlds to re-draw (all if every entry is True)
    lo: float,
    hi: float,
    geom_friction: wp.array2d(dtype=wp.vec3),   # m.geom_friction, shape (nworld, ngeom)
):
    worldid, geomid = wp.tid()
    if not reset[worldid]:
        return
    # one random stream per (seed, world, episode): reproducible and different every episode
    rng = wp.rand_init(seed[0], worldid * 7919 + episode[worldid] * 104729)
    mu = wp.randf(rng, lo, hi)
    f = geom_friction[worldid, geomid]
    geom_friction[worldid, geomid] = wp.vec3(mu, f[1], f[2])


@wp.kernel
def _bump_episode_kernel(reset: wp.array(dtype=bool), episode: wp.array(dtype=int)):
    worldid = wp.tid()
    if reset[worldid]:
        episode[worldid] = episode[worldid] + 1


def randomize(m, seed, episode, reset, lo=0.1, hi=1.0):
    """Draw a new sliding friction in [lo, hi) for every geom of the worlds marked in reset.

    Writes m.geom_friction in place (it must already have one row per world, e.g. from
    put_model(batch_sizes={"geom_friction": nworld})), so a captured CUDA graph that contains
    this call keeps working: only array CONTENTS change, never array objects.
    seed: wp.array(dtype=int, shape=(1,)); episode: wp.array(dtype=int, shape=(nworld,));
    reset: wp.array(dtype=bool, shape=(nworld,)).
    """
    nworld = reset.shape[0]
    assert m.geom_friction.shape[0] == nworld, "batch geom_friction to nworld first"
    wp.launch(_randomize_friction_kernel, dim=(nworld, m.ngeom),
              inputs=[seed, episode, reset, lo, hi], outputs=[m.geom_friction])
    wp.launch(_bump_episode_kernel, dim=nworld, inputs=[reset], outputs=[episode])


NWORLD = 4
mjm = mujoco.MjModel.from_xml_string(SLOPE)
m = mjw.put_model(mjm, batch_sizes={"geom_friction": NWORLD})
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=8, njmax=32)

seed = wp.array([7], dtype=int)
episode = wp.zeros(NWORLD, dtype=int)
all_worlds = wp.array([True] * NWORLD, dtype=bool)
some_worlds = wp.array([True, False, True, False], dtype=bool)

mu = lambda: m.geom_friction.numpy()[:, :, 0]      # (nworld, ngeom): slope and box columns
print("from the XML         mu =", mu()[:, 0])
randomize(m, seed, episode, all_worlds)
print("reset 1 (all worlds) mu =", mu()[:, 0], "| slope == box:", np.allclose(mu()[:, 0], mu()[:, 1]))
randomize(m, seed, episode, some_worlds)
mjw.reset_data(m, d, reset=some_worlds)           # physics reset of the same worlds
print("reset 2 (worlds 0, 2) mu =", mu()[:, 0], "| episodes:", episode.numpy())

# The same call inside a graph: capture once, change only the seed's CONTENT, replay.
def episode_start():
    randomize(m, seed, episode, all_worlds)
    mjw.reset_data(m, d, reset=all_worlds)
    mjw.step(m, d)

episode_start()                                    # warm up: compile every kernel first
try:
    with wp.ScopedCapture() as cap:
        episode_start()
    for s in (11, 12):
        seed.assign([s])                           # a copy into the same array, outside the graph
        wp.capture_launch(cap.graph)
        print(f"graph replay, seed {s}: mu =", mu()[:, 0])
except Exception as e:                             # older Warp CPU builds cannot capture
    print("skipped graph capture:", type(e).__name__, e)
print("device:", wp.get_device(), "| episodes:", episode.numpy())

## 10. Part II checkpoint project: Robust slide
Friction (on device), mass (set_const) and slope (tilted gravity) per world; buffers sized from a measurement rollout (`nconmax = ceil(1.5 * pooled peak nacon / nworld) + 1`, `njmax = ceil(1.5 * per-world peak nefc) + 4`); overflow checked. Uses `my_env` from the first cell.

In [ ]:
# Example 10.11 - ex11_checkpoint_robust_slide.py
"""Example 10.11 - Part II checkpoint "Robust slide": a domain-randomization study with sizing.

NWORLD boxes on a plane, each world with its own friction (my_env.randomize, on device), mass
(batch_sizes + set_const) and slope (a tilted gravity vector: for a box on an infinite plane this
is the same physics as tilting the plane). Buffers are sized from a measurement rollout (Ch 9
recipe), then the study runs and reports which worlds slid more than 0.5 m.
Run: python ex11_checkpoint_robust_slide.py [--dump-json PATH]
"""
import argparse
import json
import pathlib
import sys

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

HERE = pathlib.Path(__file__).parent if "__file__" in globals() else pathlib.Path.cwd()
sys.path.insert(0, str(HERE))
try:
    import my_env                           # notebook: written by the %%writefile cell
except ImportError:
    import my_env_step10 as my_env          # examples/10-domain-randomization/

np.set_printoptions(precision=3, suppress=True)

FLAT_BOX = """
<mujoco model="flat_box">
  <option timestep="0.005"/>
  <worldbody>
    <geom name="floor" type="plane" size="50 50 .1"/>
    <body name="box" pos="0 0 .101">
      <freejoint/>
      <geom name="box" type="box" size=".1 .1 .1" mass="1"/>
    </body>
  </worldbody>
</mujoco>
"""
parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 1024 if ON_GPU else 16
NSTEP = 200                                         # 1 s
rng = np.random.default_rng(0)

mjm = mujoco.MjModel.from_xml_string(FLAT_BOX)
box = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, "box")
DR_FIELDS = ["geom_friction", "body_mass", "body_inertia",                       # inputs
             "body_subtreemass", "body_invweight0", "dof_invweight0"]            # derived
m = mjw.put_model(mjm, batch_sizes={k: NWORLD for k in DR_FIELDS})


def make_randomized_data(nconmax, njmax):
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=nconmax, njmax=njmax)
    mjw.set_const(m, d)                             # derived mass constants, one row per world
    return d


# --- 1. draw the parameters ------------------------------------------------------------------
seed, episode = wp.array([2026], dtype=int), wp.zeros(NWORLD, dtype=int)
all_worlds = wp.array(np.ones(NWORLD, dtype=bool), dtype=bool)
my_env.randomize(m, seed, episode, all_worlds, lo=0.1, hi=1.0)     # friction, on device
mass = rng.uniform(0.5, 5.0, NWORLD)
bm, bi = m.body_mass.numpy(), m.body_inertia.numpy()
bi[:, box] *= (mass / bm[:, box])[:, None]
bm[:, box] = mass
m.body_mass.assign(bm)
m.body_inertia.assign(bi)
slope = np.radians(rng.uniform(15.0, 40.0, NWORLD))
grav = np.stack([9.81 * np.sin(slope), np.zeros(NWORLD), -9.81 * np.cos(slope)], axis=1)
m.opt.gravity = wp.array(grav, dtype=wp.vec3)                   # opt fields: replace
mu = m.geom_friction.numpy()[:, 0, 0]

# --- 2. measurement rollout with a generous budget (Ch 9 recipe) ------------------------------
d = make_randomized_data(nconmax=32, njmax=128)
max_nacon, max_nefc = 0, 0
for i in range(NSTEP):
    mjw.step(m, d)
    if i % 10 == 0:                                 # sampled reads: this run only measures
        max_nacon = max(max_nacon, int(d.nacon.numpy()[0]))
        max_nefc = max(max_nefc, int(d.nefc.numpy().max()))
nconmax = int(np.ceil(1.5 * max_nacon / NWORLD)) + 1
njmax = int(np.ceil(1.5 * max_nefc)) + 4
print(f"measured: nacon <= {max_nacon} (all worlds), nefc <= {max_nefc} (per world)")
print(f"sized:    nconmax = {nconmax} per world, njmax = {njmax}")

# --- 3. the study ------------------------------------------------------------------------------
d = make_randomized_data(nconmax=nconmax, njmax=njmax)
x0 = d.qpos.numpy()[:, 0].copy()
for _ in range(NSTEP):
    mjw.step(m, d)
wp.synchronize()
slide = d.qpos.numpy()[:, 0] - x0
pred = np.maximum(0.0, 0.5 * 9.81 * (np.sin(slope) - mu * np.cos(slope)))   # Coulomb, t = 1 s

print(f"{NWORLD} worlds: mu in [0.1, 1.0), mass in [0.5, 5) kg, slope in [15, 40) deg")
edges = [0.1, 0.3, 0.5, 0.7, 1.0]
for lo, hi in zip(edges[:-1], edges[1:]):
    sel = (mu >= lo) & (mu < hi)
    frac = (slide[sel] > 0.5).mean() if sel.any() else float("nan")
    print(f"  mu in [{lo:.1f}, {hi:.1f}): {sel.sum():4d} worlds, slid > 0.5 m: {frac:5.0%}")
print("slides when mu < tan(slope):", bool(np.all((slide > 0.05) == (mu < np.tan(slope)))))
print("max |slide - Coulomb| =", round(float(np.abs(slide - pred).max()), 3), "m")
print("corr(mass, slide - Coulomb) =", round(float(np.corrcoef(mass, slide - pred)[0, 1]), 2),
      "(small: Coulomb sliding does not depend on mass)")
print("overflow:", my_env.describe_overflow(d))

if args.dump_json:
    with open(args.dump_json, "w") as f:
        r3 = lambda a: [round(float(v), 3) for v in a]
        json.dump({"nworld": NWORLD, "nstep": NSTEP, "nconmax": nconmax, "njmax": njmax,
                   "mu": r3(mu), "slope_deg": r3(np.degrees(slope)), "mass": r3(mass),
                   "slide": r3(slide), "pred": r3(pred)}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

### Robustness histogram
The same study with many more worlds (4096 on a GPU). Your own numbers print first; the plot needs matplotlib (preinstalled on Colab). Run the Example 10.11 cell first.

In [ ]:
# Robustness histogram: the Example 10.11 study again, 4096 worlds on a GPU (64 on CPU).
# Reuses FLAT_BOX, DR_FIELDS and my_env from the Example 10.11 cell.
NW_H = 4096 if ON_GPU else 64
mjm_h = mujoco.MjModel.from_xml_string(FLAT_BOX)
box_h = mujoco.mj_name2id(mjm_h, mujoco.mjtObj.mjOBJ_BODY, "box")
m_h = mjw.put_model(mjm_h, batch_sizes={k: NW_H for k in DR_FIELDS})
d_h = mjw.make_data(mjm_h, nworld=NW_H, nconmax=8, njmax=32)
rng_h = np.random.default_rng(1)
my_env.randomize(m_h, wp.array([7], dtype=int), wp.zeros(NW_H, dtype=int),
                 wp.array(np.ones(NW_H, dtype=bool), dtype=bool), lo=0.1, hi=1.0)
mass_h = rng_h.uniform(0.5, 5.0, NW_H)
bm, bi = m_h.body_mass.numpy(), m_h.body_inertia.numpy()
bi[:, box_h] *= (mass_h / bm[:, box_h])[:, None]
bm[:, box_h] = mass_h
m_h.body_mass.assign(bm)
m_h.body_inertia.assign(bi)
slope_h = np.radians(rng_h.uniform(15.0, 40.0, NW_H))
m_h.opt.gravity = wp.array(np.stack([9.81 * np.sin(slope_h), 0 * slope_h, -9.81 * np.cos(slope_h)], 1),
                           dtype=wp.vec3)
mjw.set_const(m_h, d_h)
x0_h = d_h.qpos.numpy()[:, 0].copy()
for _ in range(200):
    mjw.step(m_h, d_h)
slide_h = d_h.qpos.numpy()[:, 0] - x0_h
mu_h = m_h.geom_friction.numpy()[:, 0, 0]
print(f"{NW_H} worlds, slid > 0.5 m: {(slide_h > 0.5).mean():.0%}, overflow:", my_env.describe_overflow(d_h))
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for lo, hi in [(0.1, 0.3), (0.3, 0.5), (0.5, 0.7), (0.7, 1.0)]:
        sel = (mu_h >= lo) & (mu_h < hi)
        ax.hist(slide_h[sel], bins=40, range=(0, 3), alpha=0.6, label=f"mu in [{lo}, {hi})")
    ax.axvline(0.5, color="k", ls="--", lw=1)
    ax.set_xlabel("slide distance after 1 s [m]"); ax.set_ylabel("worlds"); ax.legend()
    fig.tight_layout()
    plt.show()
except ImportError:
    print("skipped: matplotlib not installed")

## Exercise 10.1: a batch_sizes recipe for mass DR
Write `dr_batch_sizes(nworld)` returning the batch_sizes dict for randomizing body masses (inputs and derived outputs). The asserts check shapes and the per-world results of `set_const`.

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

SLIDER_BOX = """                      # the slider box of Example 10.5, compacted
<mujoco><worldbody>
  <body name="box" pos="0.5 0 0.5"><joint name="rail" type="slide" axis="1 0 0"/>
    <geom type="box" size=".1 .1 .1" mass="1" contype="0" conaffinity="0"/>
    <body name="payload" pos="0 0 .15"><geom type="sphere" size=".05" mass="0.5" contype="0" conaffinity="0"/></body>
  </body></worldbody>
  <actuator><motor joint="rail" gear="1"/></actuator>
</mujoco>"""

def dr_batch_sizes(nworld):
    ...  # your code: return {field_name: nworld, ...} for the inputs AND the derived outputs

NW = 3
mjm = mujoco.MjModel.from_xml_string(SLIDER_BOX)
sizes = dr_batch_sizes(NW)
m = mjw.put_model(mjm, batch_sizes=sizes)
d = mjw.make_data(mjm, nworld=NW)
assert set(sizes) >= {"body_mass", "body_subtreemass", "body_invweight0", "dof_invweight0", "actuator_acc0"}
for k in sizes:
    assert getattr(m, k).shape[0] == NW, k
bm = m.body_mass.numpy(); bm[:, 1] = [1.0, 2.0, 3.0]; m.body_mass.assign(bm)
mjw.set_const(m, d)
np.testing.assert_allclose(m.body_subtreemass.numpy()[:, 1], [1.5, 2.5, 3.5], rtol=1e-5)
np.testing.assert_allclose(m.actuator_acc0.numpy()[:, 0], 1 / np.array([1.5, 2.5, 3.5]), rtol=1e-4)
np.testing.assert_allclose(m.dof_invweight0.numpy()[:, 0], 1 / np.array([1.5, 2.5, 3.5]), rtol=1e-4)
print("exercise 10.1 passed")

In [ ]:
#@title Exercise 10.1: solution (double-click to show the code)

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

SLIDER_BOX = """                      # the slider box of Example 10.5, compacted
<mujoco><worldbody>
  <body name="box" pos="0.5 0 0.5"><joint name="rail" type="slide" axis="1 0 0"/>
    <geom type="box" size=".1 .1 .1" mass="1" contype="0" conaffinity="0"/>
    <body name="payload" pos="0 0 .15"><geom type="sphere" size=".05" mass="0.5" contype="0" conaffinity="0"/></body>
  </body></worldbody>
  <actuator><motor joint="rail" gear="1"/></actuator>
</mujoco>"""

def dr_batch_sizes(nworld):
    inputs = ["body_mass", "body_inertia"]
    derived = ["body_subtreemass", "body_invweight0", "dof_invweight0", "actuator_acc0"]
    return {k: nworld for k in inputs + derived}

NW = 3
mjm = mujoco.MjModel.from_xml_string(SLIDER_BOX)
sizes = dr_batch_sizes(NW)
m = mjw.put_model(mjm, batch_sizes=sizes)
d = mjw.make_data(mjm, nworld=NW)
assert set(sizes) >= {"body_mass", "body_subtreemass", "body_invweight0", "dof_invweight0", "actuator_acc0"}
for k in sizes:
    assert getattr(m, k).shape[0] == NW, k
bm = m.body_mass.numpy(); bm[:, 1] = [1.0, 2.0, 3.0]; m.body_mass.assign(bm)
mjw.set_const(m, d)
np.testing.assert_allclose(m.body_subtreemass.numpy()[:, 1], [1.5, 2.5, 3.5], rtol=1e-5)
np.testing.assert_allclose(m.actuator_acc0.numpy()[:, 0], 1 / np.array([1.5, 2.5, 3.5]), rtol=1e-4)
np.testing.assert_allclose(m.dof_invweight0.numpy()[:, 0], 1 / np.array([1.5, 2.5, 3.5]), rtol=1e-4)
print("exercise 10.1 passed")

## Exercise 10.2: randomize only the floor
Box friction 0.05 in the XML, slope friction 0.2 / 0.35 / 0.5 per world. Assert that the slide distances strictly decrease, the slowest moved more than 0.1 m, and nothing overflowed. Why does this work when randomizing only the box would not?

In [ ]:
# your code here (start from Example 10.4)
# assert dist[0] > dist[1] > dist[2] > 0.1
# assert not d.overflow.numpy().any()

In [ ]:
#@title Exercise 10.2: solution (double-click to show the code)

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

SLOPE = """
<mujoco><option timestep="0.005"/><worldbody>
  <geom name="slope" type="plane" size="5 5 .1" euler="0 30 0"/>
  <body name="box" pos="0.0505 0 0.08747" euler="0 30 0"><freejoint/>
    <geom name="box" type="box" size=".1 .1 .1" mass="1" friction="0.05 0.005 0.0001"/></body>
</worldbody></mujoco>"""
FLOOR_MU = [0.2, 0.35, 0.5]
mjm = mujoco.MjModel.from_xml_string(SLOPE)
floor = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_GEOM, "slope")
m = mjw.put_model(mjm, batch_sizes={"geom_friction": len(FLOOR_MU)})
d = mjw.make_data(mjm, nworld=len(FLOOR_MU), nconmax=8, njmax=32)
fr = m.geom_friction.numpy()
fr[:, floor, 0] = FLOOR_MU          # floor only; the box keeps 0.05 (max rule picks the floor)
m.geom_friction.assign(fr)
x0 = d.qpos.numpy()[:, :3].copy()
for _ in range(200):
    mjw.step(m, d)
dist = np.linalg.norm(d.qpos.numpy()[:, :3] - x0, axis=1)
print("slide:", dist.round(3))
assert dist[0] > dist[1] > dist[2] > 0.1
assert not d.overflow.numpy().any()
print("exercise 10.2 passed")

## Stretch: bounce height versus contact damping
Batch `geom_solref`, keep timeconst 0.02 s, give each world a damping ratio in [0.2, 1.0], drop the ball of Example 10.7 from 1 m and record the apex of the first bounce per world. Plot apex against dampratio and explain the shape. On a GPU, sweep both solref entries over 4096 worlds and draw a heat map.

In [ ]:
# your experiment here